## Basic validations
- `Accuracy`
- `Consistency`
- `Completeness`
- `Uniqueness`

# Accuracy

In [1]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

df = pd.read_csv("../../data/raw/diabetes_prediction_dataset.csv")

In [2]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)
print(f"Total Records: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print("\nColumn Names:", df.columns.tolist())
print("=" * 60)

DATA VALIDATION REPORT
Total Records: 100000
Total Columns: 9

Column Names: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']


In [3]:
# ============================================================
# 1. COMPLETENESS CHECK
# ============================================================
print("\n" + "=" * 60)
print("1. COMPLETENESS CHECK")
print("=" * 60)

missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

completeness_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage.round(2)
})

print("\nMissing Values per Column:")
print(completeness_df)

if missing_values.sum() == 0:
    print("\n PASSED: No missing values found!")
else:
    print(f"\n WARNING: {missing_values.sum()} total missing values found")


1. COMPLETENESS CHECK

Missing Values per Column:
                     Missing Values  Missing Percentage
gender                            0                 0.0
age                               0                 0.0
hypertension                      0                 0.0
heart_disease                     0                 0.0
smoking_history                   0                 0.0
bmi                               0                 0.0
HbA1c_level                       0                 0.0
blood_glucose_level               0                 0.0
diabetes                          0                 0.0

 PASSED: No missing values found!


In [4]:
# ============================================================
# 2. ACCURACY CHECK
# ============================================================
print("\n" + "=" * 60)
print("2. ACCURACY CHECK (Value Ranges)")
print("=" * 60)

# Define expected ranges for numeric columns
expected_ranges = {
    'age': (0, 120),
    'bmi': (10, 70),
    'HbA1c_level': (3, 15),
    'blood_glucose_level': (50, 400)
}

accuracy_issues = {}

for column, (min_val, max_val) in expected_ranges.items():
    if column in df.columns:
        out_of_range = df[(df[column] < min_val) | (df[column] > max_val)]
        accuracy_issues[column] = len(out_of_range)
        print(f"\n{column}:")
        print(f"  Expected Range: {min_val} - {max_val}")
        print(f"  Actual Range: {df[column].min()} - {df[column].max()}")
        print(f"  Out of Range Records: {len(out_of_range)}")
        if len(out_of_range) > 0:
            print(f"   WARNING: {len(out_of_range)} records outside expected range")


print("\nContinuous Column Unique Value Counts (discretization check):")
continuous_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
for col in continuous_cols:
    if col in df.columns:
        n_unique = df[col].nunique()
        print(f"  {col}: {n_unique} unique values", end="")
        if n_unique < 30:
            print(f"   WARNING: Only {n_unique} distinct values — column appears discretized")
        else:
            print()

# Check binary columns
binary_columns = ['hypertension', 'heart_disease', 'diabetes']
print("\nBinary Columns Validation:")
for col in binary_columns:
    if col in df.columns:
        unique_values = sorted(df[col].unique().tolist())
        valid = all(v in [0, 1] for v in unique_values)
        status = " Valid" if valid else " Invalid"
        print(f"  {col}: {unique_values} - {status}")

# Check gender column
if 'gender' in df.columns:
    print(f"\nGender Values: {df['gender'].unique()}")
    valid_genders = ['Male', 'Female']
    invalid_genders = [g for g in df['gender'].unique() if g not in valid_genders]
    if invalid_genders:
        count = len(df[df['gender'].isin(invalid_genders)])
        print(f"   WARNING: Invalid gender values found: {invalid_genders} ({count} records)")
    else:
        print("   Valid gender values")

# Check smoking_history column
if 'smoking_history' in df.columns:
    print(f"\nSmoking History Values: {df['smoking_history'].unique()}")
    no_info_count = (df['smoking_history'] == 'No Info').sum()
    print(f"   WARNING: 'No Info' entries act as hidden missing values: {no_info_count} records ({no_info_count/len(df)*100:.1f}%)")
    print(f"  Value Counts:\n{df['smoking_history'].value_counts()}")


2. ACCURACY CHECK (Value Ranges)

age:
  Expected Range: 0 - 120
  Actual Range: 0.08 - 80.0
  Out of Range Records: 0

bmi:
  Expected Range: 10 - 70
  Actual Range: 10.01 - 95.69
  Out of Range Records: 19

HbA1c_level:
  Expected Range: 3 - 15
  Actual Range: 3.5 - 9.0
  Out of Range Records: 0

blood_glucose_level:
  Expected Range: 50 - 400
  Actual Range: 80 - 300
  Out of Range Records: 0

Continuous Column Unique Value Counts (discretization check):
  age: 102 unique values
  bmi: 4247 unique values
  HbA1c_level: 18 unique values   WARNING: Only 18 distinct values — column appears discretized
  blood_glucose_level: 18 unique values   WARNING: Only 18 distinct values — column appears discretized

Binary Columns Validation:
  hypertension: [0, 1] -  Valid
  heart_disease: [0, 1] -  Valid
  diabetes: [0, 1] -  Valid

Gender Values: ['Female' 'Male' 'Other']

Smoking History Values: ['never' 'No Info' 'current' 'former' 'ever' 'not current']
  Value Counts:
smoking_history
No Inf

In [5]:
# 3. CONSISTENCY CHECK

print("\n" + "=" * 60)
print("3. CONSISTENCY CHECK")
print("=" * 60)

consistency_issues = {}

# Check: Young patients with chronic conditions
young_with_conditions = df[(df['age'] < 10) & ((df['hypertension'] == 1) | (df['heart_disease'] == 1))]
consistency_issues['young_with_conditions'] = len(young_with_conditions)
print(f"\nYoung Patients (<10) with Hypertension/Heart Disease: {len(young_with_conditions)}")
if len(young_with_conditions) > 0:
    print("   WARNING: Clinically unusual")
    print(young_with_conditions[['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'diabetes']].to_string(index=True))

# Age < 1 records having a smoking history is biologically impossible.
infants = df[df['age'] < 1]
consistency_issues['infants'] = len(infants)
print(f"\nPatients with Age < 1 Year: {len(infants)}")
if len(infants) > 0:
    infant_smoking = infants[infants['smoking_history'] != 'No Info']
    print(f"   WARNING: {len(infants)} infant records found — verify data integrity")
    if len(infant_smoking) > 0:
        print(f"   ERROR: {len(infant_smoking)} infants have a non-empty smoking history — biologically impossible")
        print(infant_smoking['smoking_history'].value_counts().to_string())

# Check: Diabetes patients should have higher HbA1c on average
diabetes_hba1c = df[df['diabetes'] == 1]['HbA1c_level'].mean()
non_diabetes_hba1c = df[df['diabetes'] == 0]['HbA1c_level'].mean()
print(f"\nHbA1c Level Comparison:")
print(f"  Diabetes Patients (mean): {diabetes_hba1c:.2f}")
print(f"  Non-Diabetes Patients (mean): {non_diabetes_hba1c:.2f}")
if diabetes_hba1c > non_diabetes_hba1c:
    print("   Consistent: Diabetes patients have higher HbA1c")
else:
    print("   WARNING: Unexpected HbA1c relationship")

# Check: Diabetes patients should have higher blood glucose on average
diabetes_glucose = df[df['diabetes'] == 1]['blood_glucose_level'].mean()
non_diabetes_glucose = df[df['diabetes'] == 0]['blood_glucose_level'].mean()
print(f"\nBlood Glucose Level Comparison:")
print(f"  Diabetes Patients (mean): {diabetes_glucose:.2f}")
print(f"  Non-Diabetes Patients (mean): {non_diabetes_glucose:.2f}")
if diabetes_glucose > non_diabetes_glucose:
    print("   Consistent: Diabetes patients have higher blood glucose")
else:
    print("   WARNING: Unexpected blood glucose relationship")

# Check: Very high blood glucose without diabetes diagnosis
high_glucose_no_diabetes = df[(df['blood_glucose_level'] > 300) & (df['diabetes'] == 0)]
consistency_issues['high_glucose_no_diabetes'] = len(high_glucose_no_diabetes)
print(f"\nHigh Blood Glucose (>300) without Diabetes: {len(high_glucose_no_diabetes)}")
if len(high_glucose_no_diabetes) > 0:
    print("   WARNING: May indicate mislabeling or pre-diabetic cases")

high_hba1c_no_diabetes = df[(df['HbA1c_level'] >= 7.5) & (df['diabetes'] == 0)]
consistency_issues['high_hba1c_no_diabetes'] = len(high_hba1c_no_diabetes)
print(f"\nHigh HbA1c (>=7.5%, clinical diabetes threshold) without Diabetes Label: {len(high_hba1c_no_diabetes)}")
if len(high_hba1c_no_diabetes) > 0:
    print(f"   WARNING: {len(high_hba1c_no_diabetes)} records have HbA1c at or above the diagnostic threshold but are labelled non-diabetic")
    print(f"  This may indicate pre-diabetic cases, measurement lag, or label noise")
    print(f"  HbA1c distribution in these records:")
    print(high_hba1c_no_diabetes['HbA1c_level'].value_counts().sort_index().to_string())


3. CONSISTENCY CHECK

Young Patients (<10) with Hypertension/Heart Disease: 6
       gender  age  hypertension  heart_disease smoking_history  diabetes
25508  Female  4.0             1              0         No Info         0
42836    Male  8.0             0              1     not current         0
51720    Male  7.0             1              0         current         0
60553    Male  6.0             0              1            ever         1
73654    Male  8.0             1              0         No Info         0
89531    Male  2.0             0              1         No Info         0

Patients with Age < 1 Year: 911
   ERROR: 47 infants have a non-empty smoking history — biologically impossible
smoking_history
never          37
not current     9
current         1

HbA1c Level Comparison:
  Diabetes Patients (mean): 6.93
  Non-Diabetes Patients (mean): 5.40
   Consistent: Diabetes patients have higher HbA1c

Blood Glucose Level Comparison:
  Diabetes Patients (mean): 194.09
  Non-

In [6]:
# 4. UNIQUENESS CHECK (Duplicates)
print("\n" + "=" * 60)
print("4. UNIQUENESS CHECK (Duplicates)")
print("=" * 60)

# Check for exact duplicates
duplicates = df.duplicated()
duplicate_count = duplicates.sum()

print(f"\nDuplicate Records: {duplicate_count}")

if duplicate_count > 0:
    print(f"  {duplicate_count} duplicate records found")
else:
    print("  No duplicate records found")


4. UNIQUENESS CHECK (Duplicates)

Duplicate Records: 3854
  3854 duplicate records found


In [7]:
print("\n" + "=" * 60)
print("5. DATA QUALITY SUMMARY")
print("=" * 60)

summary = {
    'Total Records': len(df),
    'Missing Values': missing_values.sum(),
    'Duplicate Records': duplicate_count,
    'BMI Outliers (>70)': accuracy_issues.get('bmi', 0),
    'Invalid Gender Values': len(df[~df['gender'].isin(['Male', 'Female'])]) if 'gender' in df.columns else 0,
    'Smoking "No Info" Entries': (df['smoking_history'] == 'No Info').sum() if 'smoking_history' in df.columns else 0,
    'Young Patients (<10) with Conditions': consistency_issues.get('young_with_conditions', 0),
    'Infant Records (<1 yr)': consistency_issues.get('infants', 0),
    'High Glucose (>300) No Diabetes': consistency_issues.get('high_glucose_no_diabetes', 0),
    'High HbA1c (>=7.5%) No Diabetes': consistency_issues.get('high_hba1c_no_diabetes', 0),
    'HbA1c Unique Values (discretization)': df['HbA1c_level'].nunique() if 'HbA1c_level' in df.columns else 'N/A',
    'Blood Glucose Unique Values (discretization)': df['blood_glucose_level'].nunique() if 'blood_glucose_level' in df.columns else 'N/A',
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Issue', 'Count'])
print("\n" + summary_df.to_string(index=False))

# Counts how many distinct issue categories are present (not total record counts)
total_issues = sum([
    missing_values.sum() > 0,
    duplicate_count > 0,
    accuracy_issues.get('bmi', 0) > 0,
    len(df[~df['gender'].isin(['Male', 'Female'])]) > 0 if 'gender' in df.columns else 0,
    consistency_issues.get('young_with_conditions', 0) > 0,
    consistency_issues.get('infants', 0) > 0,
    consistency_issues.get('high_hba1c_no_diabetes', 0) > 0,
])


5. DATA QUALITY SUMMARY

                                       Issue  Count
                               Total Records 100000
                              Missing Values      0
                           Duplicate Records   3854
                          BMI Outliers (>70)     19
                       Invalid Gender Values     18
                   Smoking "No Info" Entries  35816
        Young Patients (<10) with Conditions      6
                      Infant Records (<1 yr)    911
             High Glucose (>300) No Diabetes      0
             High HbA1c (>=7.5%) No Diabetes      0
        HbA1c Unique Values (discretization)     18
Blood Glucose Unique Values (discretization)     18
